In [9]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option('display.width', 140)

UPLOAD_DIR = '/content'
OUT_DIR = '/content'
CHART_DIR = os.path.join(OUT_DIR, 'charts')
os.makedirs(CHART_DIR, exist_ok=True)

SENTIMENT_ORDER = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
COLORS = ['#8B0000', '#E74C3C', '#95A5A6', '#27AE60', '#145A32']

In [12]:
def load_data():
    fg = pd.read_csv(os.path.join(UPLOAD_DIR, 'fear_greed_index.csv'))
    hd = pd.read_csv(os.path.join(UPLOAD_DIR, 'historical_data.csv'))

    fg['date'] = pd.to_datetime(fg['date'])
    fg = fg[['date', 'value', 'classification']].rename(
        columns={'value': 'fg_value', 'classification': 'sentiment'}
    )

    hd['dt'] = pd.to_datetime(hd['Timestamp IST'], format='%d-%m-%Y %H:%M')
    hd['date'] = hd['dt'].dt.floor('D')

    return fg, hd


def merge_data(fg, hd):
    df = hd.merge(fg, on='date', how='left')
    matched = df['sentiment'].notna().sum()
    print(f"Trades matched to a sentiment day: {matched}/{len(df)} "
          f"({matched/len(df)*100:.3f}%)")
    df = df.dropna(subset=['sentiment'])
    return df




In [10]:
fg_df, hd_df = load_data()
df = merge_data(fg_df, hd_df)

Trades matched to a sentiment day: 211218/211224 (99.997%)


In [11]:
display(df.head())

,Account,Coin,Execution Price,Size Tokens,Size USD,Side,Timestamp IST,Start Position,Direction,Closed PnL,Transaction Hash,Order ID,Crossed,Fee,Trade ID,Timestamp,dt,date,fg_value,sentiment
0,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9769,986.87,7872.16,BUY,02-12-2024 22:50,0.000000,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.345404,8.950000e+14,1.730000e+12,2024-12-02 22:50:00,2024-12-02,80.0,Extreme Greed
1,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9800,16.00,127.68,BUY,02-12-2024 22:50,986.524596,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.005600,4.430000e+14,1.730000e+12,2024-12-02 22:50:00,2024-12-02,80.0,Extreme Greed
2,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9855,144.09,1150.63,BUY,02-12-2024 22:50,1002.518996,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050431,6.600000e+14,1.730000e+12,2024-12-02 22:50:00,2024-12-02,80.0,Extreme Greed
3,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9874,142.98,1142.04,BUY,02-12-2024 22:50,1146.558564,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.050043,1.080000e+15,1.730000e+12,2024-12-02 22:50:00,2024-12-02,80.0,Extreme Greed
4,0xae5eacaf9c6b9111fd53034a602c192a04e082ed,@107,7.9894,8.73,69.75,BUY,02-12-2024 22:50,1289.488521,Buy,0.0,0xec09451986a1874e3a980418412fcd0201f500c95bac...,52017706630,True,0.003055,1.050000e+15,1.730000e+12,2024-12-02 22:50:00,2024-12-02,80.0,Extreme Greed


In [22]:
perf_df, vol_df, mix_pct_df = summary_tables(df)


=== Closed-trade performance by sentiment ===
               trades   total_pnl  avg_pnl  median_pnl  win_rate_pct
sentiment                                                           
Extreme Fear    10406   739110.25    71.03        6.39         76.22
Fear            29808  3357155.44   112.63        6.35         87.29
Neutral         18159  1292920.68    71.20        4.58         82.39
Greed           25176  2150129.27    85.40        4.93         76.89
Extreme Greed   20853  2715171.31   130.21        8.53         89.17

=== Volume & sizing by sentiment ===
               trade_count  total_volume  avg_trade_size
sentiment                                               
Extreme Fear         21400  1.144843e+08         5349.73
Fear                 61837  4.833248e+08         7816.11
Neutral              37686  1.802421e+08         4782.73
Greed                50303  2.885825e+08         5736.88
Extreme Greed        39992  1.244652e+08         3112.25

=== Long vs Short mix (% of posi

In [21]:
account_level_analysis(df)


=== Account-level ===
Total accounts: 32 | Profitable: 29 | Losing: 3
Accounts with >=20 trades in both Fear & Greed: 24
better_in
Greed    16
Fear      8
Name: count, dtype: int64


In [13]:

def summary_tables(df):
    closed = df[df['Closed PnL'] != 0].copy()
    closed['win'] = closed['Closed PnL'] > 0

    perf = closed.groupby('sentiment').agg(
        trades=('Closed PnL', 'count'),
        total_pnl=('Closed PnL', 'sum'),
        avg_pnl=('Closed PnL', 'mean'),
        median_pnl=('Closed PnL', 'median'),
        win_rate_pct=('win', lambda x: x.mean() * 100),
    ).reindex(SENTIMENT_ORDER)
    print("\n=== Closed-trade performance by sentiment ===")
    print(perf.round(2))

    vol = df.groupby('sentiment')['Size USD'].agg(
        trade_count='count', total_volume='sum', avg_trade_size='mean'
    ).reindex(SENTIMENT_ORDER)
    print("\n=== Volume & sizing by sentiment ===")
    print(vol.round(2))

    long_short = df[df['Direction'].isin(['Open Long', 'Open Short'])]
    mix = long_short.groupby(['sentiment', 'Direction']).size().unstack()
    mix_pct = (mix.div(mix.sum(axis=1), axis=0) * 100).reindex(SENTIMENT_ORDER)
    print("\n=== Long vs Short mix (% of position opens) by sentiment ===")
    print(mix_pct.round(1))

    print("\n=== Correlation checks ===")
    print("fg_value vs Closed PnL:", round(closed['fg_value'].corr(closed['Closed PnL']), 4))
    print("fg_value vs Size USD  :", round(df['fg_value'].corr(df['Size USD']), 4))

    fee_pct = df.groupby('sentiment').apply(
        lambda x: x['Fee'].sum() / x['Size USD'].sum() * 100, include_groups=False
    ).reindex(SENTIMENT_ORDER)
    print("\n=== Fee burden (% of notional volume) by sentiment ===")
    print(fee_pct.round(3))

    return perf, vol, mix_pct


def account_level_analysis(df):
    closed = df[df['Closed PnL'] != 0].copy()
    acct_pnl = closed.groupby('Account')['Closed PnL'].sum()
    print(f"\n=== Account-level ===")
    print(f"Total accounts: {len(acct_pnl)} | Profitable: {(acct_pnl > 0).sum()} "
          f"| Losing: {(acct_pnl < 0).sum()}")

    pivot = closed.groupby(['Account', 'sentiment'])['Closed PnL'].mean().unstack()
    cnt = closed.groupby(['Account', 'sentiment'])['Closed PnL'].count().unstack()
    mask = (cnt.get('Fear', 0).fillna(0) >= 20) & (cnt.get('Greed', 0).fillna(0) >= 20)
    sub = pivot[mask][['Fear', 'Greed']].dropna()
    sub['better_in'] = np.where(sub['Fear'] > sub['Greed'], 'Fear', 'Greed')
    print(f"Accounts with >=20 trades in both Fear & Greed: {len(sub)}")
    print(sub['better_in'].value_counts())


In [17]:
def make_charts(df):
    closed = df[df['Closed PnL'] != 0].copy()
    plt.rcParams.update({'font.size': 11, 'figure.facecolor': 'white'})

    def bar_chart(values, title, ylabel, fname, fmt='${:.0f}'):
        fig, ax = plt.subplots(figsize=(8, 5))
        bars = ax.bar(SENTIMENT_ORDER, values, color=COLORS)
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel(ylabel)
        for b, v in zip(bars, values):
            ax.text(b.get_x() + b.get_width() / 2, v, fmt.format(v),
                    ha='center', va='bottom' if v >= 0 else 'top')
        plt.xticks(rotation=15)
        plt.tight_layout()
        plt.savefig(os.path.join(CHART_DIR, fname), dpi=150)
        plt.close()

    bar_chart(
        closed.groupby('sentiment')['Closed PnL'].mean().reindex(SENTIMENT_ORDER),
        'Average Realized PnL per Trade by Market Sentiment', 'Avg Closed PnL (USD)',
        'avg_pnl_by_sentiment.png'
    )

    winrate = (closed.assign(win=closed['Closed PnL'] > 0)
               .groupby('sentiment')['win'].mean().reindex(SENTIMENT_ORDER) * 100)
    bar_chart(winrate, 'Win Rate by Market Sentiment', 'Win Rate (%)',
              'winrate_by_sentiment.png', fmt='{:.1f}%')

    volume = df.groupby('sentiment')['Size USD'].sum().reindex(SENTIMENT_ORDER) / 1e6
    bar_chart(volume, 'Total Trading Volume by Market Sentiment', 'Volume (Million USD)',
              'volume_by_sentiment.png', fmt='${:.0f}M')

    avg_size = df.groupby('sentiment')['Size USD'].mean().reindex(SENTIMENT_ORDER)
    bar_chart(avg_size, 'Average Trade Size by Market Sentiment', 'Avg Trade Size (USD)',
              'avg_trade_size.png')

    long_short = df[df['Direction'].isin(['Open Long', 'Open Short'])]
    mix = long_short.groupby(['sentiment', 'Direction']).size().unstack().reindex(SENTIMENT_ORDER)
    mix_pct = mix.div(mix.sum(axis=1), axis=0) * 100
    fig, ax = plt.subplots(figsize=(8, 5))
    mix_pct[['Open Long', 'Open Short']].plot(kind='bar', stacked=True, ax=ax,
                                               color=['#2E86C1', '#C0392B'])
    ax.set_title('Long vs Short Positioning by Market Sentiment', fontweight='bold')
    ax.set_ylabel('% of Position Opens')
    plt.xticks(rotation=15)
    plt.legend(title='')
    plt.tight_layout()
    plt.savefig(os.path.join(CHART_DIR, 'long_short_mix.png'), dpi=150)
    plt.close()

    print(f"\nCharts saved to {CHART_DIR}")

In [18]:
if __name__ == '__main__':
    fg, hd = load_data()
    df = merge_data(fg, hd)
    df.to_pickle(os.path.join(OUT_DIR, 'merged.pkl'))

    summary_tables(df)
    account_level_analysis(df)
    make_charts(df)

Trades matched to a sentiment day: 211218/211224 (99.997%)

=== Closed-trade performance by sentiment ===
               trades   total_pnl  avg_pnl  median_pnl  win_rate_pct
sentiment                                                           
Extreme Fear    10406   739110.25    71.03        6.39         76.22
Fear            29808  3357155.44   112.63        6.35         87.29
Neutral         18159  1292920.68    71.20        4.58         82.39
Greed           25176  2150129.27    85.40        4.93         76.89
Extreme Greed   20853  2715171.31   130.21        8.53         89.17

=== Volume & sizing by sentiment ===
               trade_count  total_volume  avg_trade_size
sentiment                                               
Extreme Fear         21400  1.144843e+08         5349.73
Fear                 61837  4.833248e+08         7816.11
Neutral              37686  1.802421e+08         4782.73
Greed                50303  2.885825e+08         5736.88
Extreme Greed        39992  1.2